##### PHASE 1 - DATA TRANSFER FROM SOURCE FILES TO DESTINATION FILES (MULTIPLE SOURCE FILES ARE GIVEN AS AN INPUT WITH PRIORITY)

In [ ]:
import lxml.etree as ET
import pandas as pd
import copy
import shutil
import os
import logging
import time
from datetime import datetime
import json

logging.basicConfig(
    level=logging.WARNING,
    format='%(levelname)s: %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler('calibration_process.log', mode='w')
    ]
)
file_logger = logging.getLogger('file_logger')
file_logger.setLevel(logging.INFO)
file_handler = logging.FileHandler('calibration_process.log', mode='w')
file_handler.setFormatter(logging.Formatter('%(levelname)s: %(message)s'))
file_logger.addHandler(file_handler)


# ─────────────────────────────────────────────────────────────────────────────
# CORE HELPER: linear interpolation / extrapolation
# ─────────────────────────────────────────────────────────────────────────────

def interp_extrap_values(src_x, src_v, dest_x):
    """
    Map src values onto the dest x-axis using:
      - Linear interpolation  when dest_x[j] is inside the src_x range.
      - Linear extrapolation  when dest_x[j] is outside (extends the slope of
                              the first two / last two src points).

    Parameters
    ----------
    src_x  : list[float]  – x-axis positions of source values  (len N, N >= 2)
    src_v  : list[float]  – corresponding source values         (len N)
    dest_x : list[float]  – x-axis positions of destination     (len M)

    Returns
    -------
    list[float] of length M  – one new value per dest_x position.

    Rules
    -----
    * Destination dimension M is NEVER changed; only the values are recomputed.
    * If src has only 1 point, every dest position gets that same value
      (no slope available).
    * src_x must be strictly increasing; caller is responsible for that.
    """
    n = len(src_x)
    result = []

    # Edge-case: single source point → constant fill
    if n == 1:
        return [src_v[0]] * len(dest_x)

    # Precompute boundary slopes for extrapolation
    slope_left  = (src_v[1]  - src_v[0])  / (src_x[1]  - src_x[0])   if src_x[1]  != src_x[0]  else 0.0
    slope_right = (src_v[-1] - src_v[-2]) / (src_x[-1] - src_x[-2])  if src_x[-1] != src_x[-2] else 0.0

    for dx in dest_x:
        # ── Left extrapolation ──────────────────────────────────────────────
        if dx <= src_x[0]:
            val = src_v[0] + slope_left * (dx - src_x[0])

        # ── Right extrapolation ─────────────────────────────────────────────
        elif dx >= src_x[-1]:
            val = src_v[-1] + slope_right * (dx - src_x[-1])

        # ── Interior linear interpolation ───────────────────────────────────
        else:
            # Find the segment [src_x[i], src_x[i+1]] that brackets dx
            i = 0
            for k in range(n - 1):
                if src_x[k] <= dx < src_x[k + 1]:
                    i = k
                    break
            span = src_x[i + 1] - src_x[i]
            t    = (dx - src_x[i]) / span if span != 0 else 0.0
            val  = src_v[i] + t * (src_v[i + 1] - src_v[i])

        result.append(val)

    return result


def _parse_floats(elements):
    """Extract float values from a list of lxml elements that have .text."""
    out = []
    for el in elements:
        if el.text and el.text.strip():
            try:
                out.append(float(el.text.strip()))
            except ValueError:
                pass
    return out


def _parse_strings(elements):
    """Extract stripped text strings from a list of lxml elements."""
    return [el.text.strip() for el in elements if el.text and el.text.strip()]


def _as_list(value):
    if isinstance(value, list):
        return value
    if value is None:
        return []
    return [value]


def _to_float_if_possible(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return v


def extract_labels_from_json(json_file):
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            payload = json.load(f)
    except FileNotFoundError:
        logging.error(f"File not found: '{json_file}'")
        return pd.DataFrame()
    except json.JSONDecodeError as e:
        logging.error(f"JSON parse error in '{json_file}': {e}")
        return pd.DataFrame()
    except Exception as e:
        logging.error(f"Unexpected error while parsing '{json_file}': {e}")
        return pd.DataFrame()

    data = []
    for key, item in payload.items():
        if not isinstance(item, dict):
            continue

        label = item.get('label') or key
        value = item.get('value')
        axes = item.get('axes', [])

        if label.endswith(('_C', '_CW', '_c')):
            data.append({'label': label, 'values': str(value)})

        elif label.endswith('_CA'):
            vals = [str(v) for v in _as_list(value)]
            data.append({'label': label, 'dimen': len(vals), 'values': vals})

        elif label.endswith(('_MAP', '_M')):
            rows = []
            for r in _as_list(value):
                if isinstance(r, list):
                    rows.append([_to_float_if_possible(v) for v in r])

            src_x = _as_list(axes[0]) if len(axes) > 0 else []
            src_y = _as_list(axes[1]) if len(axes) > 1 else []

            if not src_x and rows:
                src_x = list(range(max(len(r) for r in rows)))
            if not src_y and rows:
                src_y = list(range(len(rows)))

            data.append({
                'label': label,
                'map_values': rows,
                'x_dim': len(src_x),
                'y_dim': len(src_y),
                'x_dim_val': [_to_float_if_possible(v) for v in src_x],
                'y_dim_val': [_to_float_if_possible(v) for v in src_y],
            })

        elif label.endswith(('_T', '_CUR', '_Cur')):
            z_vals = [str(v) for v in _as_list(value)]
            x_vals = [str(v) for v in (_as_list(axes[0]) if len(axes) > 0 else [])]
            data.append({'label': label, 'x_values': x_vals,
                         'z_values': z_vals, 'x_dim': len(x_vals)})

    return pd.DataFrame(data)


def extract_labels(source_file, namespace):
    if source_file.lower().endswith('.json'):
        return extract_labels_from_json(source_file)
    return extract_labels_with_c(source_file, namespace)


# ─────────────────────────────────────────────────────────────────────────────
# XML helpers (unchanged from original)
# ─────────────────────────────────────────────────────────────────────────────

def filter_destination_file(original_dest, filtered_dest, labels_to_keep, namespace):
    tree = ET.parse(original_dest)
    root = tree.getroot()
    for sw in root.findall('.//SW-INSTANCE', namespaces=namespace):
        sn = sw.find('SHORT-NAME', namespaces=namespace)
        if sn is None or sn.text.strip() not in labels_to_keep:
            parent = sw.getparent()
            if parent is not None:
                parent.remove(sw)
    tree.write(filtered_dest, encoding='utf-8', xml_declaration=True)


def merge_updated_instances(main_dest, updated_dest, labels_to_merge, namespace):
    main_tree    = ET.parse(main_dest)
    main_root    = main_tree.getroot()
    updated_tree = ET.parse(updated_dest)
    updated_root = updated_tree.getroot()

    main_sw_map = {}
    for sw in main_root.findall('.//SW-INSTANCE', namespaces=namespace):
        sn = sw.find('SHORT-NAME', namespaces=namespace)
        if sn is not None and sn.text.strip() in labels_to_merge:
            main_sw_map[sn.text.strip()] = sw

    for sw in updated_root.findall('.//SW-INSTANCE', namespaces=namespace):
        sn = sw.find('SHORT-NAME', namespaces=namespace)
        if sn is not None and sn.text.strip() in labels_to_merge:
            old_sw = main_sw_map.get(sn.text.strip())
            if old_sw is not None:
                parent = old_sw.getparent()
                if parent is not None:
                    parent.replace(old_sw, sw)
    main_tree.write(main_dest, encoding='utf-8', xml_declaration=True)


def extract_labels_with_c(cdfx_file, namespace):
    try:
        tree = ET.parse(cdfx_file)
        root = tree.getroot()
    except ET.XMLSyntaxError as e:
        logging.error(f"XML Syntax Error while parsing '{cdfx_file}': {e}")
        return pd.DataFrame()
    except FileNotFoundError:
        logging.error(f"File not found: '{cdfx_file}'")
        return pd.DataFrame()
    except Exception as e:
        logging.error(f"Unexpected error while parsing '{cdfx_file}': {e}")
        return pd.DataFrame()

    data = []
    for sw in root.findall(".//SW-INSTANCE", namespaces=namespace):
        sn = sw.find("SHORT-NAME", namespaces=namespace)
        if sn is None or not sn.text:
            continue
        label = sn.text.strip()

        if label.endswith(('_C', '_CW', '_c')):
            v = sw.find("SW-VALUE-CONT/SW-VALUES-PHYS/V", namespaces=namespace)
            if v is not None and v.text and v.text.strip():
                data.append({'label': label, 'values': v.text.strip()})
            else:
                vt = sw.find("SW-VALUE-CONT/SW-VALUES-PHYS/VT", namespaces=namespace)
                if vt is not None and vt.text and vt.text.strip():
                    data.append({'label': label, 'values': vt.text.strip()})

        elif label.endswith('_CA'):
            dim = sw.find("SW-VALUE-CONT/SW-ARRAYSIZE/V", namespaces=namespace)
            if dim is not None and dim.text and dim.text.strip():
                vt_elements = sw.findall(".//SW-VALUE-CONT/SW-VALUES-PHYS/VT", namespaces=namespace)
                value = [vt.text.strip() for vt in vt_elements if vt.text and vt.text.strip()]
                if not value:
                    v_elements = sw.findall(".//SW-VALUE-CONT/SW-VALUES-PHYS/V", namespaces=namespace)
                    value = [v.text.strip() for v in v_elements if v.text and v.text.strip()]
                data.append({'label': label, 'dimen': dim.text, 'values': value})

        elif label.endswith(('_MAP', '_M')):
            sw_values = sw.find("SW-VALUE-CONT/SW-VALUES-PHYS", namespaces=namespace)
            if sw_values is not None:
                value_dict = {}
                for vg in sw_values.findall("VG", namespaces=namespace):
                    label_elem = vg.find("LABEL", namespaces=namespace)
                    if label_elem is not None and label_elem.text and label_elem.text.strip():
                        key = label_elem.text.strip()
                        try:
                            key = float(key)
                        except ValueError:
                            pass
                        values = _parse_floats(vg.findall("V", namespaces=namespace))
                        value_dict[key] = values
                axis = sw.findall(".//SW-AXIS-CONT", namespaces=namespace)
                x_vals, y_vals = [], []
                if axis:
                    sw_vp = axis[0].find("SW-VALUES-PHYS", namespaces=namespace)
                    if sw_vp is not None:
                        x_vals = _parse_floats(sw_vp.findall("V", namespaces=namespace))
                    try:
                        y_vals = sorted(value_dict.keys(),
                                        key=lambda x: (float(x) if isinstance(x, float) else x))
                    except TypeError:
                        y_vals = list(value_dict.keys())
                data.append({'label': label, 'values': value_dict,
                             'x_dim': len(x_vals), 'y_dim': len(y_vals),
                             'x_dim_val': x_vals, 'y_dim_val': y_vals})

        elif label.endswith(('_T', '_CUR', '_Cur')):
            axis = sw.findall(".//SW-VALUES-PHYS", namespaces=namespace)
            if axis:
                z_vals = _parse_strings(axis[0].findall("V", namespaces=namespace))
                if not z_vals:
                    z_vals = _parse_strings(axis[0].findall("VT", namespaces=namespace))
                x_vals = []
                if len(axis) > 1:
                    x_vals = _parse_strings(axis[1].findall("V", namespaces=namespace))
                    if not x_vals:
                        x_vals = _parse_strings(axis[1].findall("VT", namespaces=namespace))
                data.append({'label': label, 'x_values': x_vals,
                             'z_values': z_vals, 'x_dim': len(x_vals)})

    return pd.DataFrame(data)


def find_sw_instance(root, label, namespace):
    for sw in root.findall(".//SW-INSTANCE", namespaces=namespace):
        sn = sw.find("SHORT-NAME", namespaces=namespace)
        if sn is not None and sn.text and sn.text.strip() == label:
            return sw
    return None


def build_label_index(root, namespace):
    label_index = {}
    for sw in root.findall(".//SW-INSTANCE", namespaces=namespace):
        sn = sw.find("SHORT-NAME", namespaces=namespace)
        if sn is not None and sn.text and sn.text.strip():
            label_index[sn.text.strip()] = sw
    return label_index


# ─────────────────────────────────────────────────────────────────────────────
# MAIN UPDATE FUNCTION
# ─────────────────────────────────────────────────────────────────────────────

def override_incorrect_values(correct_cdfx_file, incorrect_cdfx_file, namespace):

    source_is_json = correct_cdfx_file.lower().endswith('.json')

    if not os.path.exists(incorrect_cdfx_file):
        if source_is_json:
            logging.error("Destination CDFX file is missing. Create it first when using JSON source input.")
            return pd.DataFrame()
        try:
            shutil.copyfile(correct_cdfx_file, incorrect_cdfx_file)
            print(f"Created: {incorrect_cdfx_file}")
            return pd.DataFrame({'label': [], 'value': [],
                                 'status': ['Destination Created by Copying Source']})
        except Exception as e:
            logging.error(f"Failed to copy source to destination: {e}")
            return pd.DataFrame()

    correct_df   = extract_labels(correct_cdfx_file, namespace)
    incorrect_df = extract_labels_with_c(incorrect_cdfx_file, namespace)

    if correct_df.empty:
        logging.error("No data extracted from the source file. Aborting update.")
        return pd.DataFrame()

    total_correct_labels     = len(correct_df)
    total_destination_labels = len(incorrect_df)
    source_labels  = set(correct_df['label'])
    dest_labels    = set(incorrect_df['label'])
    dest_only_count = len(dest_labels - source_labels)

    print(f"Source: {total_correct_labels} labels | "
          f"Destination: {total_destination_labels} labels | "
          f"Dest-only: {dest_only_count} | Log: calibration_process.log")

    merged_df = pd.merge(correct_df, incorrect_df, on='label',
                         suffixes=('_correct', '_incorrect'),
                         how='left', indicator=True)

    try:
        tree_incorrect = ET.parse(incorrect_cdfx_file)
        root_incorrect = tree_incorrect.getroot()
    except Exception as e:
        logging.error(f"Failed to parse destination CDFX file: {e}")
        return pd.DataFrame()

    if not source_is_json:
        try:
            tree_correct = ET.parse(correct_cdfx_file)
            root_correct = tree_correct.getroot()
        except Exception as e:
            logging.error(f"Failed to parse source CDFX file: {e}")
            return pd.DataFrame()
        label_index_correct = build_label_index(root_correct, namespace)
    else:
        label_index_correct = {}

    label_index_incorrect = build_label_index(root_incorrect, namespace)

    updated_data      = []
    start_time        = time.time()
    last_progress_time = start_time
    progress_interval  = 120
    processed_count   = 0
    updated_count     = 0
    no_update_needed_count = 0

    for idx, row in merged_df.iterrows():
        label = row['label']
        correct_sw   = None if source_is_json else label_index_correct.get(label)
        incorrect_sw = label_index_incorrect.get(label)

        if row['_merge'] == 'left_only':
            continue
        if incorrect_sw is None:
            continue
        if (not source_is_json) and correct_sw is None:
            continue

        processed_count += 1

        current_time = time.time()
        if current_time - last_progress_time >= progress_interval:
            elapsed_min = (current_time - start_time) / 60
            pct = 100 * processed_count / total_destination_labels
            print(f"[{datetime.now().strftime('%H:%M:%S')}] {elapsed_min:.0f}min | "
                  f"{processed_count}/{total_destination_labels} ({pct:.1f}%) | "
                  f"Updated: {updated_count} | No change: {no_update_needed_count}")
            last_progress_time = current_time

        # ── _C / _CW / _c  (scalar) ──────────────────────────────────────────
        # No dimension involved — just overwrite if different (unchanged logic).
        if label.endswith(('_C', '_CW', '_c')):
            correct_value   = row['values_correct']
            incorrect_value = row.get('values_incorrect', None)

            if not source_is_json:
                sw_unit_incorrect = incorrect_sw.find(".//SW-VALUE-CONT/UNIT-DISPLAY-NAME",
                                                      namespaces=namespace)
                sw_unit_correct   = correct_sw.find(".//SW-VALUE-CONT/UNIT-DISPLAY-NAME",
                                                    namespaces=namespace)
                if sw_unit_incorrect is not None and sw_unit_correct is not None:
                    sw_unit_incorrect.text = sw_unit_correct.text

            if correct_value != incorrect_value:
                sw_val_incorrect = incorrect_sw.find(
                    ".//SW-VALUE-CONT/SW-VALUES-PHYS/V", namespaces=namespace)
                if sw_val_incorrect is None:
                    sw_val_incorrect = incorrect_sw.find(
                        ".//SW-VALUE-CONT/SW-VALUES-PHYS/VT", namespaces=namespace)
                if sw_val_incorrect is not None:
                    sw_val_incorrect.text = f"{correct_value}"
                    updated_data.append({'label': label, 'values': correct_value,
                                         'status': 'Updated'})
                    updated_count += 1
            else:
                no_update_needed_count += 1

        # ── _CA  (1-D array) ─────────────────────────────────────────────────
        elif label.endswith('_CA'):
            """
            Strategy
            --------
            1. Read src values + their implicit x-axis (evenly-spaced indices
               0..N-1, then normalised to the dest x-range so both grids share
               the same coordinate space).

               Why indices?  _CA has no explicit SW-AXIS-CONT.  We normalise
               src indices onto [0, dest_dim-1] so the lerp/extrap treats
               positions correctly regardless of whether src is longer or
               shorter than dest.

            2. Call interp_extrap_values to produce exactly dest_dim values.

            3. Write those values back into the existing dest VT/V elements
               (dimension is preserved; no elements are added or removed).
            """
            incorrect_values_phys = incorrect_sw.find(
                "SW-VALUE-CONT/SW-VALUES-PHYS", namespaces=namespace)

            if incorrect_values_phys is None:
                file_logger.warning(f"  {label}: Missing SW-VALUES-PHYS — skipping")
                continue

            # Read source values from JSON or CDFX
            if source_is_json:
                src_values = row.get('values_correct', [])
                if isinstance(src_values, list):
                    src_str = [str(v) for v in src_values]
                elif pd.isna(src_values):
                    src_str = []
                else:
                    src_str = [str(src_values)]
            else:
                correct_values_phys = correct_sw.find(
                    "SW-VALUE-CONT/SW-VALUES-PHYS", namespaces=namespace)
                if correct_values_phys is None:
                    continue
                src_str = _parse_strings(
                    correct_values_phys.findall("VT", namespaces=namespace))
                if not src_str:
                    src_str = _parse_strings(
                        correct_values_phys.findall("V", namespaces=namespace))

            # Identify dest elements
            dest_elems = incorrect_values_phys.findall("VT", namespaces=namespace)
            if not dest_elems:
                dest_elems = incorrect_values_phys.findall("V", namespaces=namespace)

            src_n  = len(src_str)
            dest_n = len(dest_elems)

            if src_n == 0 or dest_n == 0:
                continue

            # Try numeric interpolation; fall back to nearest-neighbour for
            # string (enum) arrays where lerp has no meaning.
            try:
                src_v_float = [float(v) for v in src_str]
                numeric = True
            except ValueError:
                numeric = False

            if numeric:
                # Map src indices onto dest index range so the two grids share
                # a common coordinate space [0 .. dest_n-1].
                if src_n > 1:
                    src_x = [i * (dest_n - 1) / (src_n - 1) for i in range(src_n)]
                else:
                    src_x = [0.0]
                dest_x  = list(range(dest_n))
                new_vals = interp_extrap_values(src_x, src_v_float, dest_x)
                for i, elem in enumerate(dest_elems):
                    elem.text = f"{new_vals[i]}"
            else:
                # Nearest-neighbour for enum / string arrays
                for i, elem in enumerate(dest_elems):
                    src_idx = round(i * (src_n - 1) / max(dest_n - 1, 1))
                    src_idx = max(0, min(src_n - 1, src_idx))
                    elem.text = src_str[src_idx]

            final_vals = [e.text for e in dest_elems]
            updated_data.append({'label': label, 'dimen': dest_n,
                                  'values': final_vals, 'status': 'Updated'})
            updated_count += 1

        # ── _MAP / _M  (2-D map) ─────────────────────────────────────────────
        elif label.endswith(('_MAP', '_M')):
            """
            Strategy
            --------
            For each axis independently:
              • Read src axis values (SW-AXIS-CONT)  → src_x
              • Read dest axis values                → dest_x   (fixed length)
              • Call interp_extrap_values to get new dest axis values
                (the axis coordinates themselves change to match src meaning,
                 but count stays the same).

            For the data cells (VG / V):
              • Each VG row corresponds to one y-axis position.
              • Within a row, each V corresponds to one x-axis position.
              • We use the updated dest axis grids as the query positions so
                interp_extrap maps both axes correctly.
            """
            sw_val_incorrect = incorrect_sw.find(
                "SW-VALUE-CONT/SW-VALUES-PHYS", namespaces=namespace)
            if sw_val_incorrect is None:
                continue

            axis_incorrect = incorrect_sw.findall(".//SW-AXIS-CONT", namespaces=namespace)

            if source_is_json:
                src_row_data = []
                for src_row in row.get('map_values_correct', []):
                    if isinstance(src_row, list) and src_row:
                        parsed = []
                        for v in src_row:
                            try:
                                parsed.append(float(v))
                            except (TypeError, ValueError):
                                parsed.append(float('nan'))
                        src_row_data.append(parsed)
                if not src_row_data:
                    continue

                src_x_axis = row.get('x_dim_val_correct', [])
                src_y_axis = row.get('y_dim_val_correct', [])

                try:
                    src_x_axis = [float(v) for v in src_x_axis]
                except Exception:
                    src_x_axis = []
                try:
                    src_y_axis = [float(v) for v in src_y_axis]
                except Exception:
                    src_y_axis = []
            else:
                sw_val_correct = correct_sw.find(
                    "SW-VALUE-CONT/SW-VALUES-PHYS", namespaces=namespace)
                if sw_val_correct is None:
                    continue
                correct_vgs = sw_val_correct.findall("VG", namespaces=namespace)
                src_row_data = [_parse_floats(vg.findall("V", namespaces=namespace)) for vg in correct_vgs]
                src_row_data = [r for r in src_row_data if r]
                if not src_row_data:
                    continue

                axis_correct = correct_sw.findall(".//SW-AXIS-CONT", namespaces=namespace)
                src_x_axis = []
                src_y_axis = []
                if axis_correct:
                    svp_c = axis_correct[0].find("SW-VALUES-PHYS", namespaces=namespace)
                    if svp_c is not None:
                        src_x_axis = _parse_floats(svp_c.findall("V", namespaces=namespace))
                if len(axis_correct) > 1:
                    svp_c1 = axis_correct[1].find("SW-VALUES-PHYS", namespaces=namespace)
                    if svp_c1 is not None:
                        src_y_axis = _parse_floats(svp_c1.findall("V", namespaces=namespace))

            # ── x-axis (axis[0]) ──────────────────────────────────────────
            x_vals_new = []
            if axis_incorrect:
                svp_d = axis_incorrect[0].find("SW-VALUES-PHYS", namespaces=namespace)
                if svp_d is not None:
                    dest_x_elems = svp_d.findall("V", namespaces=namespace)
                    dest_x_axis = _parse_floats(dest_x_elems)
                    if not src_x_axis and src_row_data:
                        src_x_axis = list(range(max(len(r) for r in src_row_data)))

                    if len(src_x_axis) >= 2 and len(dest_x_axis) > 0:
                        # Normalise src positions onto dest index range
                        src_x_norm = [i * (len(dest_x_axis) - 1) / (len(src_x_axis) - 1)
                                      for i in range(len(src_x_axis))]
                        dest_pos   = list(range(len(dest_x_axis)))
                        x_vals_new = interp_extrap_values(
                            src_x_norm, src_x_axis, dest_pos)
                        for i, elem in enumerate(dest_x_elems):
                            elem.text = f"{x_vals_new[i]}"
                    elif len(src_x_axis) == 1 and dest_x_elems:
                        for elem in dest_x_elems:
                            elem.text = f"{src_x_axis[0]}"
                        x_vals_new = [src_x_axis[0]] * len(dest_x_elems)
                    else:
                        x_vals_new = dest_x_axis

            # ── y-axis (axis[1]) ──────────────────────────────────────────
            y_vals_new = []
            if len(axis_incorrect) > 1:
                svp_d1 = axis_incorrect[1].find("SW-VALUES-PHYS", namespaces=namespace)
                if svp_d1 is not None:
                    dest_y_elems = svp_d1.findall("V", namespaces=namespace)
                    dest_y_axis = _parse_floats(dest_y_elems)
                    if not src_y_axis and src_row_data:
                        src_y_axis = list(range(len(src_row_data)))

                    if len(src_y_axis) >= 2 and len(dest_y_axis) > 0:
                        src_y_norm = [i * (len(dest_y_axis) - 1) / (len(src_y_axis) - 1)
                                      for i in range(len(src_y_axis))]
                        dest_pos = list(range(len(dest_y_axis)))
                        y_vals_new = interp_extrap_values(src_y_norm, src_y_axis, dest_pos)
                        for i, elem in enumerate(dest_y_elems):
                            elem.text = f"{y_vals_new[i]}"
                    elif len(src_y_axis) == 1 and dest_y_elems:
                        for elem in dest_y_elems:
                            elem.text = f"{src_y_axis[0]}"
                        y_vals_new = [src_y_axis[0]] * len(dest_y_elems)
                    else:
                        y_vals_new = dest_y_axis

            # ── data cells (VG rows × V columns) ─────────────────────────
            incorrect_vgs = sw_val_incorrect.findall("VG", namespaces=namespace)
            src_row_count = len(src_row_data)
            dest_row_count = len(incorrect_vgs)
            if src_row_count == 0 or dest_row_count == 0:
                continue

            if src_row_count >= 2:
                src_row_norm = [i * (dest_row_count - 1) / (src_row_count - 1)
                                for i in range(src_row_count)]
            else:
                src_row_norm = [0.0]

            max_src_cols = max((len(r) for r in src_row_data), default=0)

            for row_i, dest_vg in enumerate(incorrect_vgs):
                dest_v_elems = dest_vg.findall("V", namespaces=namespace)
                dest_col_count = len(dest_v_elems)
                if dest_col_count == 0:
                    continue

                if max_src_cols >= 2:
                    src_col_norm = [j * (dest_col_count - 1) / (max_src_cols - 1)
                                    for j in range(max_src_cols)]
                elif max_src_cols == 1:
                    src_col_norm = [0.0]
                else:
                    continue

                dest_col_positions = list(range(dest_col_count))
                row_interp_vals = []
                for col_j in range(max_src_cols):
                    col_signal = [r[col_j] if col_j < len(r) else r[-1]
                                  for r in src_row_data]
                    interped = interp_extrap_values(src_row_norm, col_signal, [row_i])
                    row_interp_vals.append(interped[0])

                if max_src_cols >= 2:
                    final_col_vals = interp_extrap_values(
                        src_col_norm, row_interp_vals, dest_col_positions)
                elif max_src_cols == 1:
                    final_col_vals = [row_interp_vals[0]] * dest_col_count
                else:
                    continue

                for col_i, v_elem in enumerate(dest_v_elems):
                    v_elem.text = f"{final_col_vals[col_i]}"

                lbl_el = dest_vg.find("LABEL", namespaces=namespace)
                if lbl_el is not None and y_vals_new and row_i < len(y_vals_new):
                    lbl_el.text = f"{y_vals_new[row_i]}"

            value_dict = {}
            for vg in incorrect_vgs:
                lbl_el = vg.find("LABEL", namespaces=namespace)
                key = lbl_el.text.strip() if (lbl_el is not None and lbl_el.text) else str(id(vg))
                value_dict[key] = _parse_floats(vg.findall("V", namespaces=namespace))

            updated_data.append({'label': label, 'values': value_dict,
                                  'x_dim': len(x_vals_new), 'y_dim': len(y_vals_new),
                                  'x_dim_val': x_vals_new, 'y_dim_val': y_vals_new,
                                  'status': 'Updated'})
            updated_count += 1

        # ── _T / _CUR / _Cur  (1-D curve) ────────────────────────────────────
        elif label.endswith(('_T', '_CUR', '_Cur')):
            """
            Strategy
            --------
            Both the z-values (signal) and the x-axis values are treated as
            separate 1-D signals and mapped through interp_extrap_values.

            src x-axis  → provides the physical positions of src z-values.
            dest x-axis → provides the query positions we want z-values at.

            Result: dest dimension is kept, values are recalculated.
            """
            axis_incorrect = incorrect_sw.findall(".//SW-VALUES-PHYS", namespaces=namespace)
            if not axis_incorrect:
                continue

            # ── z-values ──────────────────────────────────────────────────
            if source_is_json:
                src_z_values = row.get('z_values_correct', [])
                src_z_str = [str(v) for v in (src_z_values if isinstance(src_z_values, list) else [src_z_values])]
            else:
                axis_correct = correct_sw.findall(".//SW-VALUES-PHYS", namespaces=namespace)
                if not axis_correct:
                    continue
                src_z_str = _parse_strings(axis_correct[0].findall("V", namespaces=namespace))
                if not src_z_str:
                    src_z_str = _parse_strings(
                        axis_correct[0].findall("VT", namespaces=namespace))

            dest_z_elems = axis_incorrect[0].findall("V", namespaces=namespace)
            if not dest_z_elems:
                dest_z_elems = axis_incorrect[0].findall("VT", namespaces=namespace)

            src_z_n  = len(src_z_str)
            dest_z_n = len(dest_z_elems)

            if src_z_n == 0 or dest_z_n == 0:
                continue

            # ── x-axis ────────────────────────────────────────────────────
            src_x_str = []
            dest_x_elems = []
            if len(axis_incorrect) > 1:
                if source_is_json:
                    src_x_values = row.get('x_values_correct', [])
                    src_x_str = [str(v) for v in (src_x_values if isinstance(src_x_values, list) else [src_x_values])]
                else:
                    src_x_str = _parse_strings(
                        axis_correct[1].findall("V", namespaces=namespace))
                    if not src_x_str:
                        src_x_str = _parse_strings(
                            axis_correct[1].findall("VT", namespaces=namespace))
                dest_x_elems = axis_incorrect[1].findall("V", namespaces=namespace)
                if not dest_x_elems:
                    dest_x_elems = axis_incorrect[1].findall("VT", namespaces=namespace)

            dest_x_n = len(dest_x_elems) if dest_x_elems else 0

            # ── Try numeric path ──────────────────────────────────────────
            try:
                src_z_float  = [float(v) for v in src_z_str]
                numeric_z    = True
            except ValueError:
                numeric_z    = False

            try:
                src_x_float  = [float(v) for v in src_x_str] if src_x_str else []
                dest_x_float = _parse_floats(dest_x_elems) if dest_x_elems else []
                numeric_x    = bool(src_x_float and dest_x_float)
            except ValueError:
                numeric_x    = False

            if numeric_z and numeric_x:
                # Use real x-axis values as the coordinate space (user's choice)
                # src_x_float  → positions of src z-values
                # dest_x_float → query positions
                if len(src_x_float) == src_z_n:
                    new_z = interp_extrap_values(src_x_float, src_z_float,
                                                 dest_x_float)
                else:
                    # Axis length mismatch between src z and src x — fall back
                    # to index normalisation
                    if src_z_n >= 2:
                        src_x_norm = [i * (dest_z_n - 1) / (src_z_n - 1)
                                      for i in range(src_z_n)]
                    else:
                        src_x_norm = [0.0]
                    new_z = interp_extrap_values(
                        src_x_norm, src_z_float, list(range(dest_z_n)))

                for i, elem in enumerate(dest_z_elems):
                    elem.text = f"{new_z[i]}"

                # Also update dest x-axis so it spans src's range properly
                # (interpolate src x onto dest grid — keeps dest count)
                if dest_x_n > 0 and len(src_x_float) >= 2:
                    if src_z_n >= 2:
                        src_x_norm2 = [i * (dest_x_n - 1) / (len(src_x_float) - 1)
                                       for i in range(len(src_x_float))]
                    else:
                        src_x_norm2 = [0.0]
                    new_x = interp_extrap_values(
                        src_x_norm2, src_x_float, list(range(dest_x_n)))
                    for i, elem in enumerate(dest_x_elems):
                        elem.text = f"{new_x[i]}"

            elif numeric_z:
                # No usable x-axis — use normalised index positions
                if src_z_n >= 2:
                    src_x_norm = [i * (dest_z_n - 1) / (src_z_n - 1)
                                  for i in range(src_z_n)]
                else:
                    src_x_norm = [0.0]
                new_z = interp_extrap_values(
                    src_x_norm, src_z_float, list(range(dest_z_n)))
                for i, elem in enumerate(dest_z_elems):
                    elem.text = f"{new_z[i]}"

            else:
                # String/enum — nearest-neighbour
                for i, elem in enumerate(dest_z_elems):
                    src_idx = round(i * (src_z_n - 1) / max(dest_z_n - 1, 1))
                    src_idx = max(0, min(src_z_n - 1, src_idx))
                    elem.text = src_z_str[src_idx]

            x_vals = [e.text for e in dest_x_elems] if dest_x_elems else []
            z_vals = [e.text for e in dest_z_elems]
            updated_data.append({'label': label, 'x_values': x_vals,
                                  'z_values': z_vals, 'x_dim': len(x_vals),
                                  'status': 'Updated'})
            updated_count += 1

    # ── Write updated tree ────────────────────────────────────────────────────
    try:
        tree_incorrect.write(incorrect_cdfx_file, encoding='utf-8',
                             xml_declaration=True)
        print(f"\nDestination file updated and saved to '{incorrect_cdfx_file}'.")
    except Exception as e:
        logging.error(f"Failed to write updates: {e}")
        return pd.DataFrame()

    total_time = (time.time() - start_time) / 60
    pct_updated  = 100 * updated_count / total_destination_labels if total_destination_labels else 0
    pct_no_change = 100 * no_update_needed_count / total_destination_labels if total_destination_labels else 0
    labels_needing_attention = total_destination_labels - updated_count - no_update_needed_count
    dest_only_count_val = len(dest_labels - source_labels)
    pct_dest_only = 100 * dest_only_count_val / total_destination_labels if total_destination_labels else 0

    print(f"\n{'='*60}")
    print(f"SUMMARY:")
    print(f"  Updated:        {updated_count:5d} ({pct_updated:.1f}%)")
    print(f"  No change:      {no_update_needed_count:5d} ({pct_no_change:.1f}%)")
    print(f"  Need attention: {labels_needing_attention:5d} "
          f"({100*labels_needing_attention/total_destination_labels if total_destination_labels else 0:.1f}%)")
    print(f"  Dest-only:      {dest_only_count_val:5d} ({pct_dest_only:.1f}%)")
    print(f"  Total:          {total_destination_labels:5d}")
    print(f"  Time:           {total_time:.1f} min")
    print(f"{'='*60}")

    return pd.DataFrame(updated_data)


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    namespace = {'autosar': 'http://autosar.org/schema/r4.0'}

    source_files     = ['A2L_HEX_parser/cal.json']  # You can also mix JSON and CDFX sources.
    destination_file = 'ADM_28.CDFX'

    all_labels_updated = set()
    all_labels         = set()
    summary            = []

    dest_df = extract_labels_with_c(destination_file, namespace)
    if dest_df.empty:
        print("No labels found in destination file. Exiting.")
        exit(1)

    all_labels  = set(dest_df['label'])
    labels_left = set(all_labels)
    print(f"Initial: {len(all_labels)} labels in destination.")

    for idx, src in enumerate(source_files):
        print(f"\n--- Processing Source {idx+1}: {src} ---")
        temp_dest = f"_temp_dest_{idx}.CDFX"
        filter_destination_file(destination_file, temp_dest, labels_left, namespace)

        src_df     = extract_labels(src, namespace)
        src_labels = set(src_df['label'])
        found_in_src = labels_left & src_labels

        updated_df     = override_incorrect_values(src, temp_dest, namespace)
        updated_labels = set(updated_df['label'])
        all_labels_updated.update(updated_labels)

        found_but_unchanged = found_in_src - updated_labels
        not_found_in_src    = labels_left - found_in_src

        merge_updated_instances(destination_file, temp_dest, updated_labels, namespace)
        os.remove(temp_dest)

        labels_left = not_found_in_src

        print(f"Source {idx+1} summary:")
        print(f"  Labels considered this round: {len(found_in_src) + len(not_found_in_src)}")
        print(f"    Updated:              {len(updated_labels)}")
        print(f"    Found but unchanged:  {len(found_but_unchanged)}")
        print(f"    Still need attention: {len(not_found_in_src)}")
        summary.append({'source': src, 'updated': len(updated_labels),
                        'found_but_unchanged': len(found_but_unchanged),
                        'still_need_attention': len(not_found_in_src),
                        'remaining': len(labels_left)})

    print("\n=== FINAL SUMMARY ===")
    for idx, s in enumerate(summary):
        print(f"Source {idx+1} ({s['source']}): Updated {s['updated']}, "
              f"Found but unchanged: {s['found_but_unchanged']}, "
              f"Still need attention: {s['still_need_attention']}, "
              f"Remaining after: {s['remaining']}")

    pct_left = 100 * len(labels_left) / len(all_labels) if all_labels else 0
    print(f"\nTotal labels in destination:      {len(all_labels)}")
    print(f"Total updated from all sources:   {len(all_labels_updated)}")
    print(f"Labels left after all sources:    {len(labels_left)} ({pct_left:.1f}%)")

    if labels_left:
        pd.DataFrame({'label': sorted(labels_left)}).to_excel(
            'needs_attention_labels.xlsx', index=False)
        print("Labels needing attention saved to needs_attention_labels.xlsx.")

Initial: 86746 labels in destination.

--- Processing Source 1: ADM_65.CDFX ---
Source: 57712 labels | Destination: 86746 labels | Dest-only: 40272 | Log: calibration_process.log


INFO: 
INFO: Destination file successfully updated and saved to '_temp_dest_0.CDFX'
INFO: ================================================================================



Destination file has been updated and saved to '_temp_dest_0.CDFX'.

SUMMARY:
  Updated:        10493 (12.1%)
  No change:      35981 (41.5%)
  Need attention: 40272 (46.4%)
  Dest-only:      40272 (46.4%)
  Total:          86746
  Time:           0.1 min
Source 1 summary:
  Labels considered this round: 86746
    Updated: 10493
    Found but unchanged: 35981
    Still need attention: 40272

--- Processing Source 2: ADM_40.CDFX ---
Source: 79852 labels | Destination: 40272 labels | Dest-only: 17359 | Log: calibration_process.log


INFO: 
INFO: Destination file successfully updated and saved to '_temp_dest_1.CDFX'
INFO: ================================================================================



Destination file has been updated and saved to '_temp_dest_1.CDFX'.

SUMMARY:
  Updated:         3012 (7.5%)
  No change:      19901 (49.4%)
  Need attention: 17359 (43.1%)
  Dest-only:      17359 (43.1%)
  Total:          40272
  Time:           0.1 min
Source 2 summary:
  Labels considered this round: 40272
    Updated: 3012
    Found but unchanged: 19901
    Still need attention: 17359

=== FINAL SUMMARY ===
Source 1 (ADM_65.CDFX): Updated 10493, Found but unchanged: 35981, Still need attention: 40272, Remaining after: 40272
Source 2 (ADM_40.CDFX): Updated 3012, Found but unchanged: 19901, Still need attention: 17359, Remaining after: 17359

Total labels in destination: 86746
Total updated from all sources: 13505
Labels left after all sources: 17359 (20.0%)

Labels still to be dealt with: 17359 (20.0% of total)
A list of labels needing attention has been saved to needs_attention_labels.xlsx.


##### PHASE 2 - EXTRACTING THE FUNCTION NAME AND VERSION OF THE LABELS THAT NEED ATTENTION

In [ ]:
"""
A2L Parser - Grouped Layout Output
Layout per group:
  Row 1 : [function_full_name] [function_name] [function_version] [label=HEADER MARKER]
  Row 2+: [function_full_name] [function_name] [function_version] [label]  <- repeated for all labels
  Row N+1, N+2: blank spacer rows
  ... next function group ...
"""

import re
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side

# ─── SET YOUR PATHS HERE ─────────────────────────────────────────────────────
excel_path  = r"C:\Users\AFO3KOR\Desktop\AI Caliberation\needs_attention_labels.xlsx"
a2l_path    = r"C:\Users\AFO3KOR\Desktop\AI Caliberation\MD1CV_P2572_MD1CE300_2_0_0.a2l"
output_path = excel_path   # Overwrites same file; change if you want a separate output
# ─────────────────────────────────────────────────────────────────────────────


def build_label_lookup(a2l_path: str):
    """Single-pass parse. Returns label->func_name and label->func_version dicts."""
    label_to_func = {}
    label_to_ver  = {}

    func_block_re = re.compile(r'/begin\s+FUNCTION',          re.IGNORECASE)
    end_func_re   = re.compile(r'/end\s+FUNCTION',            re.IGNORECASE)
    def_char_re   = re.compile(r'/begin\s+DEF_CHARACTERISTIC', re.IGNORECASE)
    end_def_re    = re.compile(r'/end\s+DEF_CHARACTERISTIC',   re.IGNORECASE)
    func_ver_re   = re.compile(r'FUNCTION_VERSION\s+"([^"]*)"', re.IGNORECASE)

    with open(a2l_path, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read()

    func_starts = [m.start() for m in func_block_re.finditer(content)]
    func_ends   = [m.end()   for m in end_func_re.finditer(content)]

    pairs, ei = [], 0
    for si in func_starts:
        while ei < len(func_ends) and func_ends[ei] <= si:
            ei += 1
        if ei < len(func_ends):
            pairs.append((si, func_ends[ei]))
            ei += 1

    for start, end in pairs:
        block = content[start:end]
        lines = block.splitlines()
        func_name = None
        for i, line in enumerate(lines):
            if func_block_re.search(line.strip()):
                for j in range(i + 1, len(lines)):
                    c = lines[j].strip()
                    if c and not c.startswith('"') and not c.startswith('/'):
                        func_name = c.split()[0]
                        break
                break
        if func_name is None:
            continue

        ver_match = func_ver_re.search(block)
        func_ver  = ver_match.group(1) if ver_match else ""

        def_match = def_char_re.search(block)
        end_match = end_def_re.search(block)
        if def_match and end_match and def_match.start() < end_match.start():
            for lbl in block[def_match.end(): end_match.start()].split():
                label_to_func[lbl] = func_name
                label_to_ver[lbl]  = func_ver

    print(f"Parsed {len(pairs)} FUNCTION blocks, {len(label_to_func)} unique labels indexed.")
    return label_to_func, label_to_ver


def build_grouped_rows(df, label_col, label_to_func, label_to_ver):
    """
    Groups labels by (function_name, function_version).
    Returns list of row dicts for the output sheet.
    Layout per group:
      - Header row  : function info + label = "--- <func_full_name> ---"
      - Label rows  : function info repeated + each label
      - 2 blank rows
    """
    df['_func']  = df[label_col].map(label_to_func).fillna("NOT FOUND")
    df['_ver']   = df[label_col].map(label_to_ver).fillna("")
    df['_full']  = df['_func'] + '  ' + df['_ver']

    rows = []
    not_found_labels = []

    # Group preserving order of first appearance
    seen_groups = {}
    for _, row in df.iterrows():
        key = (row['_func'], row['_ver'], row['_full'])
        if key not in seen_groups:
            seen_groups[key] = []
        seen_groups[key].append(row[label_col])

    for (func_name, func_ver, func_full), labels in seen_groups.items():
        if func_name == "NOT FOUND":
            not_found_labels.extend(labels)
            continue

        # Group header row — function info shown ONCE here, blank on label rows below
        rows.append({
            'function_full_name': func_full,
            'function_name':      func_name,
            'function_version':   func_ver,
            'label':              f'*** {func_full} ***',
            '_is_header': True
        })
        # One row per label — function columns left BLANK (shown only on header row above)
        for lbl in labels:
            rows.append({
                'function_full_name': '',
                'function_name':      '',
                'function_version':   '',
                'label':              lbl,
                '_is_header': False
            })
        # 2 blank spacer rows
        rows.append({'function_full_name': '', 'function_name': '', 'function_version': '', 'label': '', '_is_header': False})
        rows.append({'function_full_name': '', 'function_name': '', 'function_version': '', 'label': '', '_is_header': False})

    # Append NOT FOUND labels at the end as their own group
    if not_found_labels:
        rows.append({
            'function_full_name': 'NOT FOUND',
            'function_name':      'NOT FOUND',
            'function_version':   '',
            'label':              '*** NOT FOUND IN A2L ***',
            '_is_header': True
        })
        for lbl in not_found_labels:
            rows.append({
                'function_full_name': 'NOT FOUND',
                'function_name':      'NOT FOUND',
                'function_version':   '',
                'label':              lbl,
                '_is_header': False
            })

    missing = len(not_found_labels)
    print(f"Count of labels NOT FOUND in the A2L : {missing}")
    return rows


def write_excel(rows, output_path):
    cols = ['function_full_name', 'function_name', 'function_version', 'label']
    data = [{c: r[c] for c in cols} for r in rows]
    df_out = pd.DataFrame(data, columns=cols)
    df_out.to_excel(output_path, index=False)

    wb = load_workbook(output_path)
    ws = wb.active

    # Styles
    hdr_font   = Font(name='Arial', bold=True, color='FFFFFF')
    hdr_fill   = PatternFill('solid', start_color='1F4E79')   # dark blue
    grp_font   = Font(name='Arial', bold=True, color='FFFFFF')
    grp_fill   = PatternFill('solid', start_color='2E75B6')   # medium blue
    lbl_font   = Font(name='Arial', size=10)
    center     = Alignment(horizontal='center', vertical='center')
    left       = Alignment(horizontal='left',   vertical='center')
    thin_side  = Side(style='thin', color='AAAAAA')
    thin_border= Border(bottom=thin_side)

    # Format header row (row 1)
    for cell in ws[1]:
        cell.font      = hdr_font
        cell.fill      = hdr_fill
        cell.alignment = center

    # Map row index to _is_header flag
    is_header_map = {i + 2: r['_is_header'] for i, r in enumerate(rows)}  # +2 because row 1 = col headers

    for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
        is_grp_hdr = is_header_map.get(row_idx, False)
        for cell in row:
            if is_grp_hdr:
                cell.font      = grp_font
                cell.fill      = grp_fill
                cell.alignment = center
            else:
                cell.font      = lbl_font
                cell.alignment = left

    # Column widths
    col_widths = {'A': 45, 'B': 25, 'C': 25, 'D': 40}
    for col_letter, width in col_widths.items():
        ws.column_dimensions[col_letter].width = width

    # Freeze top row
    ws.freeze_panes = 'A2'

    wb.save(output_path)
    print(f"Saved → {output_path}")


def main():
    label_to_func, label_to_ver = build_label_lookup(a2l_path)
    df = pd.read_excel(excel_path)
    label_col = next((c for c in df.columns if c.strip().lower() == 'label'), df.columns[0])
    rows = build_grouped_rows(df, label_col, label_to_func, label_to_ver)
    write_excel(rows, output_path)


main()

Parsed 2006 FUNCTION blocks, 87316 unique labels indexed.
Labels matched: 0  |  NOT FOUND: 0
Saved → C:\Users\AFO3KOR\Desktop\AI Caliberation\needs_attention_labels.xlsx
